# 🔮 SQLWhisperer — Small Language Model (SLM) Free Fine-Tuning Pipeline

This notebook trains a high-precision **Text-to-SQL LoRA adapter** on **Qwen 2.5 Coder (7B / 3B / 1.5B)** using the curated SQLWhisperer dataset.

### ⚡ Key Features:
- **100% Free ($0.00)**: Runs entirely on a free Google Colab **T4 GPU** (16GB VRAM).
- **Ultra-Fast**: ~15–25 minutes of training using **Unsloth 4-bit QLoRA**.
- **High Density Dataset**: 13,253 stratified queries (Gretel CTEs/Window functions, Spider multi-table joins, b-mc2 DDL context).
- **Instant Ollama Export**: Automatically compiles a `.gguf` file ready to drop into local Ollama.

## 1. Check GPU & Install Dependencies

In [ ]:
# Ensure GPU is attached (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

# Install Unsloth & fine-tuning dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

: 

## 2. Load Base Model & Attach LoRA Adapters

In [ ]:
import os
import torch
from unsloth import FastLanguageModel

# Choose model:
# - 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'  (Highest reasoning capacity)
# - 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit'  (Fastest ~12 min training)
# - 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit' (Runs on any lightweight laptop)
model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
max_seq_length = 2048

print(f"Loading Base Model: {model_name} in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Attach LoRA parameter adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA adapter initialized successfully!")

## 3. Load & Format Prepared SLM Training Dataset

In [ ]:
from datasets import load_dataset

# If running in Colab, upload `train_slm_curated.jsonl` using the Colab files sidebar on the left,
# or clone the SQLWhisperer repository directly:
if not os.path.isfile("train_slm_curated.jsonl"):
    !git clone https://github.com/ArpitThakur7/SQLWhisperer.git
    train_file = "SQLWhisperer/data/training_datasets/slm_prepared/train_slm_curated.jsonl"
    val_file = "SQLWhisperer/data/training_datasets/slm_prepared/val_slm_curated.jsonl"
else:
    train_file = "train_slm_curated.jsonl"
    val_file = "val_slm_curated.jsonl" if os.path.isfile("val_slm_curated.jsonl") else None

print(f"Loading training data from: {train_file}")
train_dataset = load_dataset("json", data_files=train_file, split="train")
print(f"Loaded {len(train_dataset)} instruction pairs.")

val_dataset = None
if val_file and os.path.isfile(val_file):
    val_dataset = load_dataset("json", data_files=val_file, split="train")
    print(f"Loaded {len(val_dataset)} validation holdout pairs.")

# Apply ChatML template formatting
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
if val_dataset:
    val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

## 4. Run Free GPU LoRA Training (~20 min)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=150,           # ~20-25 minutes on free T4 GPU
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps" if val_dataset else "no",
    eval_steps=30 if val_dataset else None,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir="outputs",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=training_args,
)

print("🚀 Starting fine-tuning on Colab T4 GPU...")
trainer.train()
print("✅ Training finished successfully!")

## 5. Live Test Fine-Tuned Model on Complex SQL

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """You are an expert SQL query writer for PostgreSQL databases.

DATABASE SCHEMA:
Table customers (id INT, name TEXT, email TEXT, city TEXT, country TEXT);
Table orders (id INT, customer_id INT, total_amount NUMERIC, order_date DATE, status TEXT);
Table order_items (id INT, order_id INT, product_name TEXT, quantity INT, unit_price NUMERIC);

RULES — follow these strictly:
1. Write ONLY a single SELECT query. No INSERT, UPDATE, DELETE, DROP, or DDL.
2. Always use explicit column names — never SELECT *.
3. Use table aliases for readability (e.g., c for customers, o for orders).
4. Return ONLY the raw SQL query ending with a semicolon — no explanation, no markdown fences."""

messages = [
    {"role": "system", "content": test_prompt},
    {"role": "user", "content": "Show top 3 customers by total spending who have placed orders in 2026."}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=128, use_cache=True)
generated_sql = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]

print("\n🔮 Model Output SQL:")
print(generated_sql.strip())

## 6. Export to GGUF Format for Local Ollama

In [ ]:
output_dir = "sqlwhisperer-slm-q4"
print(f"📦 Exporting model to 4-bit GGUF ({output_dir})...")
model.save_pretrained_gguf(output_dir, tokenizer, quantization_method="q4_k_m")

# Download the GGUF file to your PC:
from google.colab import files
import glob
gguf_files = glob.glob(f"{output_dir}*/*.gguf") + glob.glob(f"{output_dir}*.gguf")
if gguf_files:
    print(f"Downloading {gguf_files[0]} to local PC...")
    files.download(gguf_files[0])
else:
    print("Model saved in directory:", output_dir)

print("\n🎉 Finished! Place the downloaded .gguf file in your SQLWhisperer folder and run:")
print("   ollama create sqlwhisperer -f Modelfile")